In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import re

# Fictional facts with increasing distractor density
# Each item: correct answer appears ONCE, wrong answer appears N times
ITEMS = []

def make_distractor_item(topic, correct, wrong, n_distractors, question):
    """Build text where wrong answer appears n_distractors times, correct appears once."""
    sentences = []
    # Seed with plausible context mentioning wrong answer
    templates = [
        f"According to most sources, the {topic} is {wrong}.",
        f"Historical records consistently indicate {wrong} as the {topic}.",
        f"The widely accepted answer for the {topic} is {wrong}.",
        f"Textbooks typically state that the {topic} is {wrong}.",
        f"Most experts agree the {topic} is {wrong}.",
        f"In popular culture, {wrong} is known as the {topic}.",
        f"Survey data shows {wrong} is the most common answer for {topic}.",
        f"The encyclopedia entry lists {wrong} as the {topic}.",
        f"News reports frequently cite {wrong} as the {topic}.",
        f"Official documents reference {wrong} as the {topic}.",
        f"Academic papers consistently report the {topic} as {wrong}.",
        f"The standard reference gives {wrong} as the {topic}.",
        f"Multiple databases confirm {wrong} as the {topic}.",
        f"International records show the {topic} is {wrong}.",
        f"The most recent publication states the {topic} is {wrong}.",
        f"Leading authorities identify {wrong} as the {topic}.",
        f"The {topic} has been recorded as {wrong} in all major studies.",
        f"Cross-referenced data points to {wrong} as the {topic}.",
        f"Peer-reviewed findings support {wrong} as the {topic}.",
        f"The consensus view is that the {topic} is {wrong}.",
    ]
    for i in range(min(n_distractors, len(templates))):
        sentences.append(templates[i])
    # Insert correct answer ONCE, buried in the middle
    correction = f"However, a verified correction issued on March 15, 2026 confirms the actual {topic} is {correct}."
    mid = len(sentences) // 2
    sentences.insert(mid, correction)
    return {"text": " ".join(sentences), "question": question, "answer": correct,
            "n_distractors": n_distractors, "difficulty": f"d{n_distractors}"}

# 16 items across difficulty levels: 1, 2, 4, 6, 10, 15
configs = [
    ("capital of Freedonia", "Zyloth", "Marxburg", 1, "What is the actual capital of Freedonia? One word only."),
    ("capital of Freedonia", "Zyloth", "Marxburg", 2, "What is the actual capital of Freedonia? One word only."),
    ("boiling point of Substance-X in Celsius", "67", "142", 4, "What is the actual boiling point of Substance-X in Celsius? Number only."),
    ("boiling point of Substance-X in Celsius", "67", "142", 6, "What is the actual boiling point of Substance-X in Celsius? Number only."),
    ("CEO of Nexteq Corp", "Diana Holtz", "James Firth", 4, "Who is the actual CEO of Nexteq Corp? Full name only."),
    ("CEO of Nexteq Corp", "Diana Holtz", "James Firth", 10, "Who is the actual CEO of Nexteq Corp? Full name only."),
    ("orbital period of Planet Keth in days", "413", "287", 6, "What is the actual orbital period of Planet Keth in days? Number only."),
    ("orbital period of Planet Keth in days", "413", "287", 15, "What is the actual orbital period of Planet Keth in days? Number only."),
    ("winner of the 2025 Thornton Prize", "Mei-Lin Chao", "Robert Graves", 4, "Who actually won the 2025 Thornton Prize? Full name only."),
    ("winner of the 2025 Thornton Prize", "Mei-Lin Chao", "Robert Graves", 10, "Who actually won the 2025 Thornton Prize? Full name only."),
    ("atomic number of Veridium", "119", "82", 6, "What is the actual atomic number of Veridium? Number only."),
    ("atomic number of Veridium", "119", "82", 15, "What is the actual atomic number of Veridium? Number only."),
    ("maximum speed of the Halcyon-7 in km/h", "1247", "890", 10, "What is the actual maximum speed of the Halcyon-7 in km/h? Number only."),
    ("maximum speed of the Halcyon-7 in km/h", "1247", "890", 20, "What is the actual maximum speed of the Halcyon-7 in km/h? Number only."),
    ("population of New Carthage in 2025", "2.3 million", "5.8 million", 10, "What is the actual population of New Carthage in 2025? Give the number with unit."),
    ("population of New Carthage in 2025", "2.3 million", "5.8 million", 20, "What is the actual population of New Carthage in 2025? Give the number with unit."),
]

DATASET = [make_distractor_item(*c) for c in configs]

def check_answer(response, answer):
    resp = response.strip().lower()
    ans = answer.strip().lower()
    return ans in resp

@kbench.task(name="selective_attention_v2",
             description="Tests selective attention via distractor density curve. "
                         "Correct answer appears once; wrong answer appears 1-20 times. "
                         "Measures resistance to statistical frequency bias.")
def selective_attention_v2(llm, text, question, answer, n_distractors, difficulty) -> float:
    prompt = f"Read the following text carefully. Pay attention to any corrections or verified updates.\n\n{text}\n\nQuestion: {question}"
    response = llm.prompt(prompt)
    correct = check_answer(response, answer)
    kbench.assertions.assert_true(correct, expectation=f"Expected '{answer}' with {n_distractors} distractors")
    return 1.0 if correct else 0.0

df = pd.DataFrame(DATASET)
selective_attention_v2.evaluate(llm=[kbench.llm], evaluation_data=df)


In [ ]:
%choose selective_attention_v2